# Notebook 4.0 — Model Training, Evaluation & Explainability
---
This notebook acts as the execution harness for the Phase 4 MLOps pipeline.

In [1]:
# Purpose: Set up project root sync paths and define constants.
# Input: None
# Output: Path variables initialized
# Expected Side Effects: sys.path modified

import sys
import logging
from pathlib import Path
import time

PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s", handlers=[logging.StreamHandler(sys.stdout)])

DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = REPORTS_DIR / "figures"

TRAIN_CLEAN_PARQUET = DATA_DIR / "train_features.parquet"
TEST_CLEAN_PARQUET = DATA_DIR / "test_features.parquet"
FEATURE_CONTRACT_PATH = REPORTS_DIR / "model_feature_shortlist.csv"

for d in [MODELS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

from src.train import (
    load_data,
    train_models,
    evaluate_model,
    build_comparison_matrix,
    plot_evaluation_curves,
    plot_confusion_matrices,
    plot_shap_summary,
    save_artifacts
)
print("ENVIRONMENT READY — All modules imported successfully.")


ENVIRONMENT READY — All modules imported successfully.


/Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Purpose: Invoke load_data() with path parameters.
# Input: TRAIN_CLEAN_PARQUET, TEST_CLEAN_PARQUET, FEATURE_CONTRACT_PATH
# Output: X_train, y_train, X_test, y_test, sorted_features
# Expected Side Effects: Dataframes loaded into memory

X_train, y_train, X_test, y_test, sorted_features = load_data(
    train_path=str(TRAIN_CLEAN_PARQUET),
    test_path=str(TEST_CLEAN_PARQUET),
    contract_path=str(FEATURE_CONTRACT_PATH),
    target_col="target"
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Number of Approved Features: {len(sorted_features)}")


X_train shape: (402754, 15)
X_test shape: (40207, 15)
Number of Approved Features: 15


In [3]:
# Purpose: Invoke train_models() to fit Random Forest and XGBoost.
# Input: X_train, y_train
# Output: Dictionary of fitted models
# Expected Side Effects: Models fit on memory, logging print statements.

start_time = time.time()
models_dict = train_models(X_train, y_train)
end_time = time.time()

print(f"Model Training Execution Time: {end_time - start_time:.2f} seconds")


Model Training Execution Time: 30.33 seconds


In [4]:
# Purpose: Iterate over fitted estimators and execute evaluate_model().
# Input: models_dict, X_train, y_train, X_test, y_test
# Output: Print ASCII summary ledger
# Expected Side Effects: Matrix CSV saved to disk.

metrics_list = []
frozen_thresholds_dict = {}
for name, model in models_dict.items():
    # Extract frozen operational threshold from train partition securely
    train_metrics = evaluate_model(model, X_train, y_train, model_name=name)
    frozen_threshold = train_metrics["Optimal-Threshold"]
    frozen_thresholds_dict[name] = frozen_threshold

    # Pass the frozen constant universally over the test split inference matrix
    metrics = evaluate_model(model, X_test, y_test, model_name=name, preset_threshold=frozen_threshold)
    metrics_list.append(metrics)

build_comparison_matrix(metrics_list, save_path=str(REPORTS_DIR / "model_comparison_matrix.csv"))


MODEL COMPARISON MATRIX
                 Model  ROC-AUC     Gini   PR-AUC  Optimal-Threshold  Accuracy  Precision   Recall  F1-Score
Random_Forest_Baseline 0.912524 0.825049 0.837491               0.41  0.823563   0.629619 0.833117  0.717213
      XGBoost_Champion 0.917128 0.834255 0.846024               0.44  0.839132   0.666232 0.803575  0.728486
[EXPORT] Comparison matrix saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/model_comparison_matrix.csv


In [5]:
# Purpose: Invoke plot_evaluation_curves()
# Input: models_dict, X_test, y_test, FIGURES_DIR
# Output: None
# Expected Side Effects: Artifact saved directly to reports/figures/

plot_evaluation_curves(models_dict, X_test, y_test, save_dir=str(FIGURES_DIR))

plot_confusion_matrices(models_dict, X_test, y_test, save_dir=str(FIGURES_DIR), frozen_thresholds=frozen_thresholds_dict)
# Output: None
# Expected Side Effects: Artifact saved directly to reports/figures/
plot_confusion_matrices(models_dict, X_test, y_test, save_dir=str(FIGURES_DIR), frozen_thresholds=frozen_thresholds_dict)


[PLOT] Evaluation curves saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/model_evaluation_curves.png
[PLOT] Confusion matrices saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/confusion_matrices.png
[PLOT] Confusion matrices saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/confusion_matrices.png


In [6]:
# Purpose: Invoke plot_shap_summary() targeting the fitted XGBoost Champion model.
# Input: models_dict, X_train, sorted_features, FIGURES_DIR
# Output: None
# Expected Side Effects: SHAP artifact saved to disk.

xgb_champion = models_dict["XGBoost_Champion"]
shap_save_path = str(FIGURES_DIR / "shap_summary_plot.png")

plot_shap_summary(
    model=xgb_champion, 
    X_train=X_train, 
    feature_names=sorted_features, 
    save_path=shap_save_path, 
    max_display_samples=1000
)

[SHAP] SHAP summary plot saved -> /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/reports/figures/shap_summary_plot.png


### Black-Box SHAP Decomposition & Actionable Business Rules

Based on the SHAP topological layout and institutional risk parameters, we establish the following concrete credit underwriting insights:

1. **Primary Risk Driver Identification:**
   - **`LTV_Utilization_Quotient_log1p`:** Heavily dominates the global impact ranking. Extreme outliers in this quotient signal immediate over-leverage.
   - **`risk_factor_above_threshold_daily_count`:** High frequency acts as a compounding penalty layer, confirming sustained risky behavioral patterns rather than one-off spikes.

2. **Directional Impairment Analysis:**
   - Elevated `LTV_Utilization_Quotient_log1p` directly increases the log-odds of default exponentially, creating steep risk cliffs as utilization approaches maximum collateral thresholds.
   - Gas premium spending metrics indicate whether the user operates in a hurried (high risk) or methodical (low risk) archetype.

3. **Actionable Risk Policy Thresholds:**
   - **Hard Cap Monitoring:** Establish an automated circuit-breaker for wallets where `LTV_Utilization_Quotient_log1p` combined with `risk_factor_above_threshold_daily_count` breaches the 95th percentile.
   - **Dynamic LTV Limits:** Scale back maximum allowable LTV on a wallet-by-wallet basis dynamically when predictive scores cross the dynamically calculated macro F1 optimal threshold.

In [7]:
# Purpose: Invoke save_artifacts()
# Input: models_dict, X_train, MODELS_DIR
# Output: None
# Expected Side Effects: Models serialized, cold-validation gate run, tokens logged.

save_artifacts(models_dict, X_train, save_dir=str(MODELS_DIR))

[SAVE] Model Random_Forest_Baseline saved to /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/models/random_forest_baseline.joblib
[VALIDATE] Executing cold-reload validation for Random_Forest_Baseline...
[VALIDATE] Cold-reload validation passed successfully for Random_Forest_Baseline.
[SAVE] Model XGBoost_Champion saved to /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/models/xgboost_champion.joblib
[VALIDATE] Executing cold-reload validation for XGBoost_Champion...
[VALIDATE] Cold-reload validation passed successfully for XGBoost_Champion.
